# 🇵🇭 Philippine Address Parsing & Normalization Pipeline — TF-IDF Edition

**What changed from v1 (rapidfuzz-only)**

| | v1 — rapidfuzz WRatio | v2 — TF-IDF + rapidfuzz fallback |
|---|---|---|
| City/province matching | O(N) loop per row | Batch sparse matrix multiply |
| Barangay matching | rapidfuzz token loop | rapidfuzz token_set_ratio (scoped) |
| Medium-conf. fallback | rapidfuzz WRatio | rapidfuzz token_set_ratio |
| Extra dependency | — | `scikit-learn` |
| Est. speed — fuzzy stage (10k rows) | ~60 s | ~3–5 s |

**Architecture**

```
Raw addresses
     │
     ▼
Stage 1 ── Alias normalization        (ph_address_alias_extended_v4.csv)
     │
     ▼
Stage 2 ── TF-IDF batch match         (city + province, vectorized)
     │         └── fallback ────────  rapidfuzz token_set_ratio
     │
     ▼
Stage 3 ── Confidence gate            (high → fast path · low → API)
     │                    │
     ▼                    ▼
  dim_location        Stage 4 ── Nominatim OSM API verify (aggressive / strict)
  fast lookup              │
     │                    ▼
     └──────── Stage 5 ──────── Output split
                               │                │
                     verified_addresses.xlsx   invalid_addresses.xlsx
                               │
                         combined_addresses.xlsx
```

**Key accuracy rules — unchanged from v1**
- Alias pre-expansion before any matcher (handles CDO, SJDM, QC, etc.)
- Province anchoring prevents substring collision (San Jose × 3 provinces)
- Disambiguation: ambiguous city requires province corroboration
- Nominatim API is the final ground-truth verifier for low-confidence rows
- `API_QUERY_MODE = 'aggressive'` enables typo-tolerant fallback query chains


## Cell 0 — Environment check

Verifies all required packages. `scikit-learn` is the **only new dependency** vs v1.

```bash
pip install scikit-learn
```


In [10]:
import importlib, sys

REQUIRED = {
    'pandas'      : 'pandas',
    'numpy'       : 'numpy',
    'rapidfuzz'   : 'rapidfuzz',
    'sklearn'     : 'sklearn',
    'scipy'       : 'scipy',
    'tqdm'        : 'tqdm',
    'openpyxl'    : 'openpyxl',
    'xlsxwriter'  : 'xlsxwriter',
}

missing = []
for pkg_label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'n/a')
        print(f'  ✅  {pkg_label:<14} {ver}')
    except ImportError:
        print(f'  ❌  {pkg_label:<14} NOT FOUND')
        missing.append(pkg_label)

if missing:
    print(f'\n⚠️  Install missing packages:')
    print(f'    pip install {" ".join(missing)}')
else:
    print('\n✅  All dependencies satisfied — ready to run.')


  ✅  pandas         3.0.1
  ✅  numpy          2.4.3
  ✅  rapidfuzz      3.14.3
  ✅  sklearn        1.8.0
  ✅  scipy          1.17.1
  ✅  tqdm           4.67.3
  ✅  openpyxl       3.1.5
  ✅  xlsxwriter     3.2.9

✅  All dependencies satisfied — ready to run.


## Cell 1 — Configuration

Edit the paths and thresholds here. Everything else in the pipeline reads from these variables.

| Variable | Default | Meaning |
|---|---|---|
| `CITY_SCORE_HIGH` | 88 | Auto-accept without API call |
| `CITY_SCORE_LOW` | 65 | Minimum to even consider a city match |
| `PROV_SCORE_HIGH` | 88 | Auto-accept province |
| `PROV_SCORE_LOW` | 60 | Minimum province score |
| `TFIDF_CUTOFF` | 0.35 | Minimum cosine similarity (0–1) for TF-IDF to produce a candidate |
| `TFIDF_FALLBACK_CUTOFF` | 0.20 | Below this, skip TF-IDF and fall straight to rapidfuzz |
| `USE_API` | `True` | Set `False` for fuzzy-only mode (faster, less accurate) |
| `API_QUERY_MODE` | `'aggressive'` | `'strict'` = minimal queries, `'aggressive'` = typo-tolerant fallbacks |
| `MAX_ROWS` | `None` | Set an integer (e.g. `100`) to process only a slice for testing |


In [11]:
from pathlib import Path

# ── File paths ───────────────────────────
BASE_DIR = Path("..") / ".." / "data"   # notebook is in update_address/notebooks

INPUT_FILE   = str(BASE_DIR / "sample"  / "sample_org_address.xlsx")
DIM_LOCATION = str(BASE_DIR / "mapping" / "dim_location_20260316_v2.csv")
ALIAS_FILE   = str(BASE_DIR / "utils"   / "ph_address_alias_extended_v4.csv")
OUT_VERIFIED  = str(BASE_DIR / "output" / "verified_addresses.xlsx")
OUT_INVALID   = str(BASE_DIR / "output" / "invalid_addresses.xlsx")

# ── Batch input (unchanged) ───────────────────────────────────────────────────
input_paths = [
    Path('../../data/sample/Structured_Philippine_Addresses_Unmatched/') / f'Structured_Philippine_Addresses_Unmatched_part_{i:03d}.xlsx'
    for i in range(1, 11)   # Adjust range to cover your batches, e.g. range(1, 101)
]

# ── Fuzzy-match thresholds (0–100) ───────────────────────────────────
CITY_SCORE_HIGH  = 88
CITY_SCORE_LOW   = 65
PROV_SCORE_HIGH  = 88
PROV_SCORE_LOW   = 60
BRGY_SCORE_MIN   = 75

# ── TF-IDF thresholds (0.0–1.0 cosine similarity) ─────────────────────
TFIDF_CUTOFF          = 0.35   # accept TF-IDF candidate above this
TFIDF_FALLBACK_CUTOFF = 0.20   # below this → skip to rapidfuzz directly

# ── Nominatim (OSM) API settings ───────────────────────────────────
NOMINATIM_URL = 'https://nominatim.openstreetmap.org/search'
NOMINATIM_UA  = 'ph-address-pipeline/1.0 (research)'
BATCH_DELAY   = 1.1
MAX_RETRIES   = 2

# ── Run controls ─────────────────────────────────────────────────
USE_API        = True
API_QUERY_MODE = 'aggressive'   # 'strict' | 'aggressive'
MAX_ROWS       = None

# ── Quick path check ───────────────────────────────────────────────
if input_paths:
    print(f'  ℹ️  Batch mode enabled: {len(input_paths)} input files')
    for p in input_paths:
        status = '✅' if p.exists() else '❌  NOT FOUND'
        print(f'  {status}  {p}')
else:
    status = '✅' if Path(INPUT_FILE).exists() else '❌  NOT FOUND'
    print(f'  {status}  {INPUT_FILE}')

for f in [DIM_LOCATION, ALIAS_FILE]:
    status = '✅' if Path(f).exists() else '❌  NOT FOUND'
    print(f'  {status}  {f}')
print(f'  ℹ️  API_QUERY_MODE = {API_QUERY_MODE}')
print(f'  ℹ️  TFIDF_CUTOFF = {TFIDF_CUTOFF}  |  TFIDF_FALLBACK_CUTOFF = {TFIDF_FALLBACK_CUTOFF}')


  ℹ️  Batch mode enabled: 10 input files
  ✅  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_001.xlsx
  ✅  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_002.xlsx
  ✅  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_003.xlsx
  ✅  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_004.xlsx
  ✅  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_005.xlsx
  ✅  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_006.xlsx
  ✅  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_007.xlsx
  ✅  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_

## Cell 2 — Imports & logging setup

In [12]:
import re
import time
import logging
import urllib.request
import urllib.parse
import json
from typing import Optional

import numpy as np
import pandas as pd
from IPython.display import display
from rapidfuzz import fuzz, process
from scipy.sparse import issparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')


14:34:35  INFO      Imports OK


## Cell 3 — Load reference data

Loads `dim_location` (42 000+ PSGC barangay rows) and `ph_address_alias` (499 alias pairs), then builds fast in-memory lookup lists for both TF-IDF and rapidfuzz.


In [13]:
def _read_csv_with_fallback(path: str) -> pd.DataFrame:
    encodings = ['utf-8', 'utf-8-sig', 'cp1252', 'latin1']
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last_error = e
            log.warning(f'Failed reading {path} with encoding={enc}; trying next fallback')
    raise UnicodeDecodeError(
        getattr(last_error, 'encoding', 'utf-8'),
        getattr(last_error, 'object', b''),
        getattr(last_error, 'start', 0),
        getattr(last_error, 'end', 1),
        f'Unable to decode {path} with tried encodings: {encodings}'
    )


def _fix_mojibake(text: str) -> str:
    # Correct common UTF-8/cp1252 mojibake sequences seen in PH names.
    if not isinstance(text, str):
        return text
    return (
        text
        .replace('Ã±', 'ñ')
        .replace('Ã‘', 'Ñ')
        .replace('Ã©', 'é')
        .replace('Ã‰', 'É')
    )


def load_reference(dim_path: str, alias_path: str):
    log.info('Loading dim_location ...')
    dim = _read_csv_with_fallback(dim_path)
    str_cols = ['region_name', 'province_name', 'city_name', 'barangay_name']
    for c in str_cols:
        dim[c] = dim[c].fillna('').astype(str).map(_fix_mojibake).str.upper().str.strip()

    log.info('Loading alias table ...')
    alias = _read_csv_with_fallback(alias_path)
    alias['pattern'] = alias['pattern'].fillna('').astype(str).map(_fix_mojibake).str.upper().str.strip()
    alias['replacement'] = alias['replacement'].fillna('').astype(str).map(_fix_mojibake).str.upper().str.strip()

    cities    = sorted(x for x in dim['city_name'].unique().tolist() if x)
    provinces = sorted(x for x in dim['province_name'].unique().tolist() if x)
    regions   = sorted(x for x in dim['region_name'].unique().tolist() if x)
    barangays = sorted(x for x in dim['barangay_name'].unique().tolist() if x)

    log.info(
        f'dim_location: {len(dim):,} rows | '
        f'{len(cities)} cities | {len(provinces)} provinces | '
        f'{len(regions)} regions | {len(barangays):,} barangays'
    )
    return dim, alias, cities, provinces, regions, barangays


dim, alias_df, cities, provinces, regions, barangays = load_reference(
    DIM_LOCATION, ALIAS_FILE
)

print('\n── Sample dim_location ──')
display(dim.head(3))
print('\n── Sample aliases ──')
display(alias_df.head(10))


14:34:35  INFO      Loading dim_location ...
14:34:35  WARNING   Failed reading ..\..\data\mapping\dim_location_20260316_v2.csv with encoding=utf-8; trying next fallback
14:34:35  WARNING   Failed reading ..\..\data\mapping\dim_location_20260316_v2.csv with encoding=utf-8-sig; trying next fallback
14:34:35  INFO      Loading alias table ...
14:34:35  WARNING   Failed reading ..\..\data\utils\ph_address_alias_extended_v4.csv with encoding=utf-8; trying next fallback
14:34:35  WARNING   Failed reading ..\..\data\utils\ph_address_alias_extended_v4.csv with encoding=utf-8-sig; trying next fallback
14:34:35  INFO      dim_location: 42,011 rows | 1426 cities | 83 provinces | 18 regions | 25,843 barangays



── Sample dim_location ──


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,AGTANGAO,14,1,1,1
1,1400101002,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,ANGAD,14,1,1,2
2,1400101003,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,BAÑACAO,14,1,1,3



── Sample aliases ──


,pattern,replacement
0,BRGY,BARANGAY
1,BRGY.,BARANGAY
2,BRG,BARANGAY
3,BRG.,BARANGAY
4,BARRIO,BARANGAY
5,BARRIO.,BARANGAY
6,POB,POBLACION
7,POB.,POBLACION
8,POBL,POBLACION
9,POBL.,POBLACION


## Cell 4 — Stage 1: Alias normalization

Builds a **single-pass compiled regex** from all 499 alias pairs (sorted longest-first so multi-word patterns like `CITY OF MARIKINA` fire before the single-token `MARIKINA`).

A post-pass deduplication step catches chained-alias artifacts like `CITY CITY` or `BARANGAY BARANGAY`.


In [14]:
def build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (p.strip(), r.strip())
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if isinstance(p, str) and p.strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))
    replace_map = {p: r for p, r in pairs}
    pattern  = '|'.join(re.escape(p) for p, _ in pairs)
    compiled = re.compile(r'\b(' + pattern + r')\b')
    return compiled, replace_map


def normalize_alias(text: str, compiled_re, replace_map: dict) -> str:
    text = str(text).upper().strip()
    text = re.sub(r'\s+', ' ', text)
    text = compiled_re.sub(lambda m: replace_map.get(m.group(0), m.group(0)), text)
    text = re.sub(
        r'\b(CITY|BARANGAY|STREET|AVENUE|BOULEVARD|VILLAGE|PROVINCE)\s+\1\b',
        r'\1', text, flags=re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', text).strip()


compiled_re, replace_map = build_alias_regex(alias_df)
log.info(f'Alias regex built from {len(alias_df)} patterns')

test_cases = [
    'sapang palay sjdm',
    '112 Ballecer st Zone South Signal Village Taguig',
    '55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City',
    'loilo City',
    'carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL',
]
print(f'{"ORIGINAL":<55}  {"NORMALIZED"}')
print('─' * 110)
for t in test_cases:
    print(f'{t:<55}  {normalize_alias(t, compiled_re, replace_map)}')


14:34:36  INFO      Alias regex built from 504 patterns


ORIGINAL                                                 NORMALIZED
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
sapang palay sjdm                                        SAPANG PALAY SAN JOSE DEL MONTE CITY
112 Ballecer st Zone South Signal Village Taguig         112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City  55 DAWN STREET SOUTH KIM VIEW PARK SSS VILLAGE CONCEPCIONH 2 MARIKINA CITY
loilo City                                               LOILO CITY
carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL  CARMENCDOC CARMEN CAGAYAN DE ORO CITY (CAPITAL) MISAMIS ORIENTAL


## Cell 5 — Stage 2: TF-IDF index + matching helpers

### How TF-IDF replaces the rapidfuzz loop

**v1 (old):** `process.extractOne(query, cities)` is called once per row inside a Python loop — O(N×M) total comparisons.

**v2 (new):**
1. At startup, `TfidfVectorizer` converts all city and province names into character n-gram vectors (built once, reused for every batch).
2. `NearestNeighbors` with cosine distance finds the closest PSGC name via a single sparse matrix multiply across **all rows at once**.
3. Only rows whose TF-IDF cosine score falls below `TFIDF_CUTOFF` fall back to `rapidfuzz.token_set_ratio` for edge-case recovery.

### Why `token_set_ratio` replaces `WRatio` for the fallback

`token_set_ratio` ignores token order and extra tokens — ideal for long messy address strings where the city name is buried inside house numbers and street names.

### Limitations of TF-IDF (mitigated by the pipeline)
- Short abbreviations (`CDO`, `QC`) produce almost no n-grams → handled by alias pre-expansion in Cell 4
- Substring collision (`SAN JOSE` × 3 provinces) → handled by province-anchored disambiguation in Cell 6


In [15]:
# ── Build TF-IDF indexes (once at startup) ─────────────────────────────────────
log.info('Building TF-IDF indexes ...')
_t0 = time.time()

_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), min_df=1)

# Fit on cities + provinces combined so the vocabulary covers both
_all_names = cities + provinces
_tfidf.fit(_all_names)

_city_matrix = _tfidf.transform(cities)       # shape: (n_cities,  vocab)
_prov_matrix = _tfidf.transform(provinces)    # shape: (n_provs,   vocab)

_city_nn = NearestNeighbors(n_neighbors=1, metric='cosine', algorithm='brute', n_jobs=-1)
_city_nn.fit(_city_matrix)

_prov_nn = NearestNeighbors(n_neighbors=1, metric='cosine', algorithm='brute', n_jobs=-1)
_prov_nn.fit(_prov_matrix)

log.info(f'TF-IDF indexes ready in {time.time()-_t0:.2f}s  '
         f'(vocab={len(_tfidf.vocabulary_):,} | '
         f'cities={len(cities)} | provinces={len(provinces)})')


# ── Single-query TF-IDF helper ────────────────────────────────────────────────────
def tfidf_match(
    query: str,
    choices: list,
    nn_index: NearestNeighbors,
    cutoff: float = TFIDF_CUTOFF,
) -> tuple[Optional[str], float]:
    """
    Match a single query string against a pre-built NearestNeighbors TF-IDF index.
    Returns (best_match_string, cosine_similarity_0_to_100) or (None, 0).
    """
    if not query:
        return None, 0.0
    vec = _tfidf.transform([query])
    distances, indices = nn_index.kneighbors(vec)
    cosine_sim = float(1.0 - distances[0][0])   # convert distance → similarity
    if cosine_sim < cutoff:
        return None, round(cosine_sim * 100, 1)
    return choices[int(indices[0][0])], round(cosine_sim * 100, 1)


def tfidf_token_scan(
    tokens: list,
    choices: list,
    nn_index: NearestNeighbors,
    cutoff: float = TFIDF_CUTOFF,
) -> tuple[Optional[str], float]:
    """Try each token with TF-IDF; return the highest-similarity result."""
    best_m, best_s = None, 0.0
    for tok in tokens:
        if len(tok) < 3:
            continue
        m, s = tfidf_match(tok, choices, nn_index, cutoff=cutoff)
        if s > best_s:
            best_m, best_s = m, s
    return best_m, best_s


# ── rapidfuzz fallback (token_set_ratio replaces WRatio from v1) ───────────────
def best_match(
    query: str,
    choices: list,
    scorer=fuzz.token_set_ratio,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """rapidfuzz fallback for edge cases TF-IDF misses."""
    if not query:
        return None, 0
    result = process.extractOne(query, choices, scorer=scorer,
                                score_cutoff=score_cutoff)
    return (result[0], int(result[1])) if result else (None, 0)


def token_scan(
    tokens: list,
    choices: list,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """rapidfuzz token scan fallback."""
    best_s, best_m = 0, None
    for tok in tokens:
        if len(tok) < 3:
            continue
        m, s = best_match(tok, choices, score_cutoff=score_cutoff)
        if s > best_s:
            best_s, best_m = s, m
    return best_m, best_s


# ── Unified matcher: TF-IDF primary, rapidfuzz fallback ──────────────────────────
def match_name(
    query: str,
    tokens: list,
    choices: list,
    nn_index: NearestNeighbors,
    score_cutoff_pct: int = 0,
) -> tuple[Optional[str], int]:
    """
    Try TF-IDF first on full query, then on tokens.
    Fall back to rapidfuzz token_set_ratio if TF-IDF cosine < TFIDF_FALLBACK_CUTOFF.
    Returns (match, score_0_to_100).
    """
    # TF-IDF: full string
    m_full, s_full = tfidf_match(query, choices, nn_index, cutoff=TFIDF_FALLBACK_CUTOFF)
    # TF-IDF: token scan
    m_tok, s_tok = tfidf_token_scan(tokens, choices, nn_index, cutoff=TFIDF_FALLBACK_CUTOFF)

    # Take best TF-IDF result
    if s_full >= s_tok:
        tfidf_m, tfidf_s = m_full, s_full
    else:
        tfidf_m, tfidf_s = m_tok, s_tok

    # If TF-IDF is confident enough, return it
    if tfidf_m and tfidf_s >= (TFIDF_CUTOFF * 100):
        return tfidf_m, int(tfidf_s)

    # Fallback: rapidfuzz token_set_ratio
    rf_full, rf_s_full = best_match(query, choices, score_cutoff=score_cutoff_pct)
    rf_tok,  rf_s_tok  = token_scan(tokens, choices, score_cutoff=score_cutoff_pct)

    if rf_s_full >= rf_s_tok:
        rf_m, rf_s = rf_full, rf_s_full
    else:
        rf_m, rf_s = rf_tok, rf_s_tok

    # Return whichever is higher between TF-IDF partial hit and rapidfuzz
    if tfidf_s >= rf_s and tfidf_m:
        return tfidf_m, int(tfidf_s)
    return rf_m, rf_s


# ── Smoke test ─────────────────────────────────────────────────────────────────────
smoke = [
    ('QUEZON CITY', cities, _city_nn),
    ('ILOCOS SUR', provinces, _prov_nn),
    ('TAGUIG', cities, _city_nn),
    ('SOUTH SIGNAL', cities, _city_nn),   # should score low
    ('CDO', cities, _city_nn),            # short abbrev — alias should have expanded this first
]
print(f'{"Query":<25}  {"TF-IDF match":<38}  Score  Method')
print('─' * 85)
for q, lst, nn in smoke:
    toks = [t for t in re.split(r'[\s,]+', q) if len(t) >= 3]
    m, s = match_name(q, toks, lst, nn, score_cutoff_pct=0)
    method = 'tfidf' if s >= TFIDF_CUTOFF * 100 else 'rapidfuzz-fallback'
    print(f'{q:<25}  {str(m):<38}  {s:<5}  {method}')


14:34:36  INFO      Building TF-IDF indexes ...
14:34:36  INFO      TF-IDF indexes ready in 0.04s  (vocab=2,639 | cities=1426 | provinces=83)


Query                      TF-IDF match                            Score  Method
─────────────────────────────────────────────────────────────────────────────────────
QUEZON CITY                QUEZON CITY                             100    tfidf
ILOCOS SUR                 ILOCOS SUR                              100    tfidf
TAGUIG                     CITY OF TAGUIG                          79     tfidf
SOUTH SIGNAL               SOUTH UPI                               78     tfidf
CDO                        CORDON                                  66     tfidf


## Cell 6 — Stage 2 + 3: Core address matching & confidence gate

### What changed from v1
- `match_name()` (TF-IDF + rapidfuzz fallback) replaces all direct `best_match()` / `token_scan()` calls for city and province resolution.
- Barangay matching still uses scoped rapidfuzz `token_set_ratio` (barangay list is already city-scoped, so TF-IDF batch gain is marginal there).
- All other logic — province anchoring, disambiguation, barangay context scoring, confidence gate — is identical to v1.

### Confidence levels
| Level | Condition | Action |
|---|---|---|
| `high` | city score ≥ 88 | Accept immediately — no API |
| `medium` | city score 65–87 | Send to Nominatim for verification |
| `low` | no city, province score < 88 | Send to Nominatim |
| `none` | nothing matched | Mark invalid |


In [16]:
def match_address(
    address_norm: str,
    dim: pd.DataFrame,
    cities: list, provinces: list, regions: list, barangays: list,
    city_high: int = CITY_SCORE_HIGH,
    city_low: int  = CITY_SCORE_LOW,
    prov_high: int = PROV_SCORE_HIGH,
    prov_low: int  = PROV_SCORE_LOW,
) -> dict:
    result = dict(
        city_name=None, city_code=None, city_score=0,
        province_name=None, province_code=None, province_score=0,
        region_name=None, region_code=None,
        barangay_name=None, barangay_code=None,
        psgc_10_digit=None,
        confidence='none', needs_api=False,
    )

    tokens = [t for t in re.split(r'[,\s]+', address_norm) if len(t) >= 3]

    # ── Province hint (TF-IDF primary + rapidfuzz fallback) ────────────────────
    prov_hint, prov_hint_score = match_name(
        address_norm, tokens, provinces, _prov_nn, score_cutoff_pct=prov_low
    )

    # ── City hint (TF-IDF primary + rapidfuzz fallback) ───────────────────────
    city_hint, city_hint_score = match_name(
        address_norm, tokens, cities, _city_nn, score_cutoff_pct=city_low
    )

    def _name_overlaps_address(name: str, text: str) -> bool:
        skip = {'CITY', 'MUNICIPALITY', 'OF', 'PROVINCE'}
        words = [w for w in str(name).split() if w not in skip and len(w) >= 4]
        return any(w in text for w in words)

    generic_brgy_terms = {
        'PUROK', 'POBLACION', 'ZONE', 'CENTRO', 'PROPER',
        'BARANGAY', 'SITIO', 'VILLA', 'DISTRICT', 'AREA'
    }

    def _best_barangay_row(brgy_name: Optional[str], brgy_raw_score: int, source: str):
        if not brgy_name:
            return None, -1, -1
        subset = dim[dim['barangay_name'] == brgy_name]
        if subset.empty:
            return None, -1, -1

        best_row, best_adjusted, best_bonus = None, -1, -1
        for _, row in subset.iterrows():
            bonus = 0
            if city_hint and row['city_name'] == city_hint:
                bonus += 25
            if prov_hint and row['province_name'] == prov_hint:
                bonus += 20
            if _name_overlaps_address(row['city_name'], address_norm):
                bonus += 10
            if _name_overlaps_address(row['province_name'], address_norm):
                bonus += 8
            adjusted = brgy_raw_score + bonus
            if adjusted > best_adjusted:
                best_adjusted, best_bonus, best_row = adjusted, bonus, row

        brgy_upper = str(brgy_name).upper().strip()
        brgy_words = [w for w in brgy_upper.split() if w]
        is_generic       = brgy_upper in generic_brgy_terms
        is_short_single  = len(brgy_words) == 1 and len(brgy_upper) < 7
        looks_placeholder = brgy_upper.startswith('BARANG')

        if source == 'token' and best_bonus < 10:
            return None, -1, -1
        if is_generic and best_bonus < 15:
            return None, -1, -1
        if (is_short_single or looks_placeholder) and best_bonus < 20:
            return None, -1, -1
        if len(brgy_words) == 1 and source == 'full' and best_bonus < 10:
            return None, -1, -1

        return best_row, best_adjusted, best_bonus

    # ── PRIORITY 1: Barangay-first with context-aware candidate scoring ──────────
    brgy_full, brgy_score_full = best_match(address_norm, barangays, score_cutoff=BRGY_SCORE_MIN)
    brgy_tok,  brgy_score_tok  = token_scan(tokens, barangays, score_cutoff=BRGY_SCORE_MIN)

    row_full, adj_full, bonus_full = _best_barangay_row(brgy_full, brgy_score_full, source='full')
    row_tok,  adj_tok,  bonus_tok  = _best_barangay_row(brgy_tok,  brgy_score_tok,  source='token')

    if max(adj_full, adj_tok) >= BRGY_SCORE_MIN:
        brgy_row = row_full if adj_full >= adj_tok else row_tok
        result.update(
            city_name=brgy_row['city_name'],       city_code=int(brgy_row['city_code']),
            city_score=95,
            province_name=brgy_row['province_name'], province_code=int(brgy_row['province_code']),
            province_score=95,
            region_name=brgy_row['region_name'],   region_code=int(brgy_row['region_code']),
            barangay_name=brgy_row['barangay_name'],
            barangay_code=int(brgy_row['barangay_code']),
            psgc_10_digit=int(brgy_row['psgc_10_digit']),
            confidence='high', needs_api=False,
        )
        return result

    # ── PRIORITY 2: City / province fallback ────────────────────────────────────────
    prov_match, prov_score = prov_hint, prov_hint_score
    city_match, city_score = city_hint, city_hint_score

    # Strict disambiguation: ambiguous city requires province corroboration
    if city_match:
        candidate_provs = dim[dim['city_name'] == city_match]['province_name'].unique()
        if len(candidate_provs) > 1:
            if not (prov_match and prov_match in candidate_provs
                    and prov_score >= prov_low):
                city_score = 0
                city_match = None

    if city_match:
        subset = dim[dim['city_name'] == city_match]
        if prov_match and prov_score >= prov_low:
            sub2 = subset[subset['province_name'] == prov_match]
            if not sub2.empty:
                subset = sub2
        first = subset.iloc[0]
        result.update(
            city_name=city_match,         city_code=int(first['city_code']),
            city_score=city_score,
            province_name=first['province_name'], province_code=int(first['province_code']),
            province_score=prov_score,
            region_name=first['region_name'],     region_code=int(first['region_code']),
        )
    elif prov_match and prov_score >= prov_high:
        subset = dim[dim['province_name'] == prov_match]
        first = subset.iloc[0]
        result.update(
            province_name=prov_match, province_code=int(first['province_code']),
            province_score=prov_score,
            region_name=first['region_name'], region_code=int(first['region_code']),
        )

    # ── Confidence classification ────────────────────────────────────────────────────────
    cs, ps = result['city_score'], result['province_score']
    if cs >= city_high:
        result['confidence'], result['needs_api'] = 'high', False
    elif cs >= city_low:
        result['confidence'], result['needs_api'] = 'medium', True
    elif result['province_name'] and ps >= prov_high:
        result['confidence'], result['needs_api'] = 'medium', True
    else:
        result['confidence'], result['needs_api'] = 'low', True

    return result


# ── Regression tests ─────────────────────────────────────────────────────────────────
test_addresses = [
    '19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY',
    '112 Ballecer st Zone South Signal Village Taguig',
]
for idx, raw in enumerate(test_addresses, 1):
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans  = match_address(addr, dim, cities, provinces, regions, barangays)
    print(f'Test {idx}')
    print(f'  Input      : {raw}')
    print(f'  Normalized : {addr}')
    print(f'  Barangay   : {ans["barangay_name"]}')
    print(f'  City       : {ans["city_name"]}')
    print(f'  Province   : {ans["province_name"]}')
    print(f'  Confidence : {ans["confidence"]}')
    print('─' * 80)


Test 1
  Input      : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Normalized : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Barangay   : None
  City       : QUEZON
  Province   : QUEZON
  Confidence : high
────────────────────────────────────────────────────────────────────────────────
Test 2
  Input      : 112 Ballecer st Zone South Signal Village Taguig
  Normalized : 112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
  Barangay   : SOUTH SIGNAL VILLAGE
  City       : CITY OF TAGUIG
  Province   : METRO MANILA
  Confidence : high
────────────────────────────────────────────────────────────────────────────────


## Cell 7 — Run Stage 1–3 on all input rows

Loads the input file, applies alias normalization, then runs the TF-IDF matcher on every row. Results are split into `high_conf` (no API needed) and `low_conf` (needs Nominatim verification).

> When `USE_API=True`, all rows are queued for API double-check regardless of fuzzy confidence.


In [17]:
import time

t_start = time.time()

if input_paths:
    existing_paths = [p for p in input_paths if p.exists()]
    missing_paths = [p for p in input_paths if not p.exists()]

    for mp in missing_paths:
        log.warning(f'Missing batch file: {mp}')

    if not existing_paths:
        raise FileNotFoundError('No batch files found in input_paths.')

    log.info(f'Loading batch files: {len(existing_paths)} found')
    frames = []
    for p in existing_paths:
        tmp = pd.read_excel(p)
        tmp['_batch_file'] = p.name
        frames.append(tmp)
    df_input = pd.concat(frames, ignore_index=True)
else:
    log.info(f'Loading {INPUT_FILE} ...')
    df_input = pd.read_excel(INPUT_FILE)

if MAX_ROWS:
    df_input = df_input.iloc[:MAX_ROWS]
    log.info(f'Limiting to {MAX_ROWS} rows (MAX_ROWS is set)')

addr_col = df_input.columns[0]
log.info(f'Address column: "{addr_col}"  |  Total rows: {len(df_input):,}')

# ── Stage 1: alias normalization ────────────────────────────────────────────────
log.info('Stage 1: alias normalization ...')
df_input['address_normalized'] = df_input[addr_col].apply(
    lambda x: normalize_alias(str(x), compiled_re, replace_map)
)

# ── Stage 2-3: TF-IDF match + confidence gate ─────────────────────────────────
log.info('Stage 2-3: TF-IDF matching + confidence gate ...')
records = []

for _, row in tqdm(df_input.iterrows(), total=len(df_input),
                   desc='TF-IDF match', unit='row'):
    rec = {
        'original_address'  : row[addr_col],
        'address_normalized': row['address_normalized'],
        'api_status': 'pending' if USE_API else 'skipped',
        'osm_city': None, 'osm_province': None, 'osm_region': None,
        'osm_display': None, 'osm_lat': None, 'osm_lon': None,
    }
    if '_batch_file' in row:
        rec['batch_file'] = row['_batch_file']
    rec.update(match_address(
        row['address_normalized'],
        dim, cities, provinces, regions, barangays,
    ))
    records.append(rec)

if USE_API:
    high_conf = []
    low_conf  = records
else:
    high_conf = [r for r in records if not r['needs_api']]
    low_conf  = [r for r in records if r['needs_api']]

t_fuzzy = time.time() - t_start
log.info(f'TF-IDF pass done in {t_fuzzy:.2f}s')
if USE_API:
    log.info(f'  Queued for API verification : {len(low_conf):,} (all rows)')
else:
    log.info(f'  High confidence (no API): {len(high_conf):,}')
    log.info(f'  Needs API verification  : {len(low_conf):,}')

preview_cols = [
    'original_address', 'address_normalized',
    'city_name', 'province_name', 'confidence', 'city_score'
]
if records and 'batch_file' in records[0]:
    preview_cols = ['batch_file'] + preview_cols

df_preview = pd.DataFrame(records)[preview_cols]
display(df_preview)


14:34:37  INFO      Loading batch files: 10 found
14:34:38  INFO      Address column: "Original_Address"  |  Total rows: 10,000
14:34:38  INFO      Stage 1: alias normalization ...
14:34:38  INFO      Stage 2-3: TF-IDF matching + confidence gate ...


TF-IDF match:   0%|          | 0/10000 [00:00<?, ?row/s]

KeyboardInterrupt: 

In [ ]:
# Snapshot TF-IDF outputs before API mutates records
pre_api_df = pd.DataFrame([dict(r) for r in records])
print(f'Captured pre_api_df rows: {len(pre_api_df):,}')


## Cell 8 — Stage 4: Nominatim API verification

When `USE_API=True`, **all rows** are sent to the [OpenStreetMap Nominatim API](https://nominatim.org/) for double-checking.
The API result is re-matched against `dim_location` and becomes the final location source.

`API_QUERY_MODE` controls fallback query strategy:
- `'strict'` → original + normalized queries only
- `'aggressive'` → typo-tolerant cleanup + city-hint candidate queries

> **OSM usage policy:** 1 request per second maximum. `BATCH_DELAY = 1.1s` keeps us within policy.
> Estimated API time: `row_count × 1.1s` (plus retries).


In [ ]:
def _nominatim_query(address: str) -> Optional[dict]:
    params = urllib.parse.urlencode({
        'q': address + ', Philippines',
        'format': 'json',
        'addressdetails': 1,
        'limit': 1,
        'countrycodes': 'ph',
    })
    req = urllib.request.Request(
        f'{NOMINATIM_URL}?{params}',
        headers={'User-Agent': NOMINATIM_UA}
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.loads(r.read())
        return data[0] if data else None
    except Exception as e:
        log.debug(f'Nominatim error: {e}')
        return None


def _osm_extract(result: dict) -> dict:
    addr = result.get('address', {})
    city_raw = (
        addr.get('city') or addr.get('town') or
        addr.get('municipality') or addr.get('county') or ''
    )
    return {
        'osm_city':    city_raw.upper().strip(),
        'osm_province': (addr.get('state_district') or addr.get('province') or '').upper().strip(),
        'osm_region':   (addr.get('state') or '').upper().strip(),
        'osm_lat':      result.get('lat'),
        'osm_lon':      result.get('lon'),
        'osm_display':  result.get('display_name', ''),
    }


def _clear_location_fields(row: dict):
    row.update(
        city_name=None, city_code=None, city_score=0,
        province_name=None, province_code=None, province_score=0,
        region_name=None, region_code=None,
        barangay_name=None, barangay_code=None,
        psgc_10_digit=None,
    )


def _should_assign_barangay(row: dict) -> bool:
    text = f"{row.get('original_address', '')} {row.get('address_normalized', '')}".upper()
    if re.search(r'\b(BRGY|BRGY\.|BRG|BRG\.|BGY|BARANGAY)\b', text):
        return True
    tokens = [t for t in re.split(r'[^A-Z0-9]+', text) if t]
    detail_cues = {
        'ST', 'STREET', 'RD', 'ROAD', 'AVE', 'AVENUE', 'BLVD', 'BOULEVARD',
        'DR', 'DRIVE', 'LANE', 'LN', 'VILLAGE', 'SUBD', 'SUBDIVISION',
        'PHASE', 'BLOCK', 'BLK', 'LOT', 'PUROK', 'ZONE', 'SITIO',
        'BUILDING', 'BLDG', 'UNIT', 'FLOOR', 'TOWER', 'EXT', 'EXTENSION'
    }
    if any(tok in detail_cues for tok in tokens):
        return True
    if len(tokens) <= 5:
        return False
    return True


def _candidate_queries(row: dict, cities: list, query_mode: str = 'aggressive') -> list:
    original   = str(row.get('original_address') or '').strip()
    normalized = str(row.get('address_normalized') or '').strip()
    candidates = []

    def _add(q: str):
        q = str(q or '').strip(' ,')
        if q and q not in candidates:
            candidates.append(q)

    _add(original)
    _add(normalized)

    if query_mode == 'strict':
        return candidates[:6]

    cleaned = re.sub(r'[^A-Z0-9\s,]', ' ', normalized.upper())
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    _add(cleaned)

    no_house_num = re.sub(r'\b\d+[A-Z]?\b', ' ', cleaned)
    no_house_num = re.sub(r'\s+', ' ', no_house_num).strip()
    _add(no_house_num)

    for m in re.findall(r'\b([A-Z]+(?:\s+[A-Z]+){0,4}\sCITY)\b', cleaned):
        _add(m)

    city_hint, city_score = best_match(cleaned, cities, score_cutoff=82)
    if city_hint:
        _add(city_hint)

    if row.get('province_name'):
        for q in candidates[:]:
            if q.endswith('CITY') or q.startswith('CITY OF'):
                _add(f'{q}, {row.get("province_name")}')

    toks = [t for t in re.split(r'[\s,]+', cleaned) if t]
    if len(toks) >= 4:
        _add(' '.join(toks[-6:]))
        _add(' '.join(toks[-4:]))

    return candidates[:12]


def _query_with_fallbacks(row: dict, cities: list, query_mode: str = 'aggressive') -> Optional[dict]:
    for q in _candidate_queries(row, cities, query_mode=query_mode):
        for _ in range(MAX_RETRIES + 1):
            osm_raw = _nominatim_query(q)
            if osm_raw:
                return osm_raw
            time.sleep(BATCH_DELAY)
    return None


def api_verify_batch(
    rows: list, dim: pd.DataFrame,
    cities: list, provinces: list, regions: list,
    query_mode: str = 'aggressive',
) -> list:
    mode = str(query_mode).strip().lower()
    if mode not in {'strict', 'aggressive'}:
        mode = 'aggressive'
    log.info(f'Sending {len(rows)} rows to Nominatim ... (mode={mode})')
    out = []
    for row in tqdm(rows, desc='API verify', unit='addr'):
        osm_raw = _query_with_fallbacks(row, cities, query_mode=mode)

        if not osm_raw:
            _clear_location_fields(row)
            row.update(confidence='none', api_status='no_result', needs_api=False)
            out.append(row)
            time.sleep(BATCH_DELAY)
            continue

        osm = _osm_extract(osm_raw)
        row.update(osm)

        c_match, c_score = (None, 0)
        p_match, p_score = (None, 0)

        if osm['osm_city']:
            c_match, c_score = best_match(osm['osm_city'], cities, score_cutoff=CITY_SCORE_LOW)
        if osm['osm_province']:
            p_match, p_score = best_match(osm['osm_province'], provinces, score_cutoff=PROV_SCORE_LOW)

        matched = False

        if c_match and c_score >= CITY_SCORE_LOW:
            subset = dim[dim['city_name'] == c_match]
            candidate_provs = subset['province_name'].unique()
            if len(candidate_provs) > 1:
                if p_match and p_match in candidate_provs and p_score >= PROV_SCORE_LOW:
                    sub2 = subset[subset['province_name'] == p_match]
                    subset = sub2 if not sub2.empty else subset.iloc[0:0]
                else:
                    subset = subset.iloc[0:0]
            elif p_match and p_score >= PROV_SCORE_LOW:
                sub2 = subset[subset['province_name'] == p_match]
                if not sub2.empty:
                    subset = sub2

            if not subset.empty:
                first = subset.iloc[0]
                row.update(
                    city_name=c_match, city_code=int(first['city_code']), city_score=int(c_score),
                    province_name=first['province_name'], province_code=int(first['province_code']),
                    province_score=max(int(p_score), PROV_SCORE_LOW) if p_match else 0,
                    region_name=first['region_name'], region_code=int(first['region_code']),
                    confidence='api_verified', api_status='matched', needs_api=False,
                )

                if _should_assign_barangay(row):
                    city_brgy = dim[dim['city_name'] == c_match]['barangay_name'].tolist()
                    toks = [t for t in re.split(r'[,\s]+', row.get('address_normalized', '')) if len(t) >= 3]
                    brgy_match, brgy_score = token_scan(toks, city_brgy, score_cutoff=BRGY_SCORE_MIN)
                    if brgy_match:
                        brgy_row = dim[
                            (dim['city_name'] == c_match) &
                            (dim['barangay_name'] == brgy_match)
                        ].iloc[0]
                        row.update(
                            barangay_name=brgy_match,
                            barangay_code=int(brgy_row['barangay_code']),
                            psgc_10_digit=int(brgy_row['psgc_10_digit']),
                        )
                else:
                    row.update(barangay_name=None, barangay_code=None, psgc_10_digit=None)

                matched = True

        if not matched and p_match and p_score >= PROV_SCORE_LOW:
            subset = dim[dim['province_name'] == p_match]
            if not subset.empty:
                first = subset.iloc[0]
                row.update(
                    city_name=None, city_code=None, city_score=0,
                    province_name=p_match, province_code=int(first['province_code']),
                    province_score=int(p_score),
                    region_name=first['region_name'], region_code=int(first['region_code']),
                    barangay_name=None, barangay_code=None, psgc_10_digit=None,
                    confidence='api_verified_province', api_status='province_only', needs_api=False,
                )
                matched = True

        if not matched:
            _clear_location_fields(row)
            row.update(confidence='none', api_status='unmatched', needs_api=False)

        out.append(row)
        time.sleep(BATCH_DELAY)

    return out


# ── Run API stage ──────────────────────────────────────────────────────────────────────
if USE_API and low_conf:
    t_api_start = time.time()
    low_conf = api_verify_batch(
        low_conf, dim, cities, provinces, regions,
        query_mode=API_QUERY_MODE,
    )
    log.info(f'API stage done in {time.time() - t_api_start:.1f}s')
else:
    if not USE_API:
        log.info('USE_API=False — skipping Nominatim stage')
    else:
        log.info('No rows available for API verification')


## Cell 9 — Stage 5a: Merge results & determine validity

- **`USE_API=True`**: valid only if API verification resolved city/province (`api_status` in `matched`, `province_only`)
- **`USE_API=False`**: fuzzy-only validity (city resolved OR province with high score)


In [ ]:
all_records = low_conf if USE_API else (high_conf + low_conf)
out_df = pd.DataFrame(all_records)

if USE_API:
    out_df['is_valid'] = (
        out_df['api_status'].isin(['matched', 'province_only']) &
        (
            out_df['city_name'].notna() |
            out_df['province_name'].notna()
        )
    )
else:
    out_df['is_valid'] = (
        out_df['city_name'].notna() |
        (
            out_df['province_name'].notna() &
            (out_df['province_score'] >= PROV_SCORE_HIGH)
        )
    )

print('Confidence distribution:')
print(out_df['confidence'].value_counts().to_string())
if 'api_status' in out_df.columns:
    print('\nAPI status distribution:')
    print(out_df['api_status'].value_counts().to_string())
print(f'\nValid   : {out_df["is_valid"].sum():,}')
print(f'Invalid : {(~out_df["is_valid"]).sum():,}')
print(f'Total   : {len(out_df):,}')


In [ ]:
# City-only sanity check: barangay should remain blank when not explicit
city_only_mask = out_df['original_address'].astype(str).str.lower().isin(['vigan ilocos sur', 'loilo city'])
display(out_df.loc[city_only_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])


In [ ]:
# Detailed-address check: barangay should be populated when appropriate
detail_mask = out_df['original_address'].astype(str).str.contains(
    'Batasan|South Signal|Kim View|sapang palay', case=False, na=False
)
display(out_df.loc[detail_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])


In [ ]:
# Before-vs-after change report (TF-IDF baseline vs API-final)
compare_cols = ['city_name', 'province_name', 'confidence', 'api_status']
base = pre_api_df[['original_address'] + compare_cols].copy()
base = base.rename(columns={
    'city_name': 'city_before', 'province_name': 'province_before',
    'confidence': 'confidence_before', 'api_status': 'api_status_before',
})
final = out_df[['original_address'] + compare_cols].copy()
final = final.rename(columns={
    'city_name': 'city_after', 'province_name': 'province_after',
    'confidence': 'confidence_after', 'api_status': 'api_status_after',
})
change_df = base.merge(final, on='original_address', how='inner')
changed_mask = (
    change_df['city_before'].fillna('') != change_df['city_after'].fillna('')
) | (
    change_df['province_before'].fillna('') != change_df['province_after'].fillna('')
) | (
    change_df['confidence_before'].fillna('') != change_df['confidence_after'].fillna('')
) | (
    change_df['api_status_before'].fillna('') != change_df['api_status_after'].fillna('')
)
changed_rows = change_df[changed_mask].reset_index(drop=True)
print(f'Rows changed after API verification: {len(changed_rows):,} / {len(change_df):,}')
display(changed_rows)


In [ ]:
# Compact delta view
compact = changed_rows[[
    'original_address', 'city_before', 'city_after',
    'province_before', 'province_after',
    'api_status_before', 'api_status_after'
]].copy()
compact['original_address'] = compact['original_address'].astype(str).str.slice(0, 70)
display(compact)

mask = changed_rows['original_address'].astype(str).str.contains('General Espino South Signal', case=False, na=False)
if mask.any():
    print('General Espino South Signal change:')
    display(changed_rows.loc[mask, [
        'original_address', 'city_before', 'city_after',
        'province_before', 'province_after',
        'confidence_before', 'confidence_after',
        'api_status_before', 'api_status_after'
    ]])


## Cell 10 — Stage 5b: Export to Excel

Creates three output files:
- `output/verified/verified_addresses.xlsx` — green tab
- `output/invalid/invalid_addresses.xlsx` — red tab
- `output/combined_addresses.xlsx` — dark blue tab, includes PSGC/confidence/OSM diagnostics + `status` column


In [ ]:
from pathlib import Path

out_root = Path(BASE_DIR) / 'output'
out_verified_dir = out_root / 'verified'
out_invalid_dir  = out_root / 'invalid'
out_verified_dir.mkdir(parents=True, exist_ok=True)
out_invalid_dir.mkdir(parents=True, exist_ok=True)

VERIFIED_FILE = str(out_verified_dir / 'verified_addresses.xlsx')
INVALID_FILE  = str(out_invalid_dir  / 'invalid_addresses.xlsx')
COMBINED_FILE = str(out_root / 'combined_addresses.xlsx')

BASE_EXPORT_COLS = [
    'Original_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
]

base_df = out_df.copy()
base_df = base_df.rename(columns={
    'original_address': 'Original_Address',
    'barangay_name': 'barangay',
    'city_name': 'city',
    'province_name': 'province',
})
for c in BASE_EXPORT_COLS:
    if c not in base_df.columns:
        base_df[c] = None

verified_df = base_df[base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)
invalid_df  = base_df[~base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)

combined_cols = BASE_EXPORT_COLS + [
    'psgc_10', 'confidence', 'city_score', 'province_score',
    'osm_display', 'osm_lat', 'osm_lon', 'status',
]
combined_df = base_df.copy()
combined_df['psgc_10'] = combined_df.get('psgc_10_digit')
combined_df['status']  = np.where(combined_df['is_valid'], 'verified', 'invalid')
for c in combined_cols:
    if c not in combined_df.columns:
        combined_df[c] = None
combined_df = combined_df[combined_cols].reset_index(drop=True)


def write_xlsx(df: pd.DataFrame, path: str, sheet_name: str,
               tab_color: str, font_color: str = '#000000'):
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        wb = writer.book
        ws = writer.sheets[sheet_name]
        ws.set_tab_color(tab_color)
        hdr_fmt = wb.add_format({
            'bold': True, 'bg_color': tab_color,
            'font_color': font_color,
            'border': 1, 'text_wrap': True,
            'font_name': 'Arial', 'font_size': 10,
        })
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, hdr_fmt)
            if df.empty:
                max_w = len(col) + 4
            else:
                col_max = df[col].dropna().astype(str).str.len().max()
                if pd.isna(col_max):
                    col_max = len(col)
                max_w = max(int(col_max), len(col)) + 4
            ws.set_column(i, i, min(int(max_w), 42))
        ws.freeze_panes(1, 0)
    log.info(f'Wrote {len(df):,} rows → {path}')


write_xlsx(verified_df, VERIFIED_FILE, 'Verified', '#00B050')
write_xlsx(invalid_df,  INVALID_FILE,  'Invalid',  '#FF0000', font_color='#FFFFFF')
write_xlsx(combined_df, COMBINED_FILE, 'Combined', '#1F4E78', font_color='#FFFFFF')

elapsed = time.time() - t_start
print(f'\nTotal elapsed: {elapsed:.1f}s  ({elapsed/60:.2f} min)')
print('Output files:')
print(f'  ✅  {VERIFIED_FILE}  ({len(verified_df):,} rows)')
print(f'  ⚠️   {INVALID_FILE}   ({len(invalid_df):,} rows)')
print(f'  [combined]  {COMBINED_FILE}  ({len(combined_df):,} rows)')


## Cell 11 — Results summary

Quick inline preview of both output tables for a final sanity check.

In [ ]:
print('═' * 80)
print('  PIPELINE SUMMARY')
print('═' * 80)
total = len(out_df)
n_v   = len(verified_df)
n_i   = len(invalid_df)
print(f'  Input rows        : {total:>8,}')
print(f'  Verified          : {n_v:>8,}  ({100*n_v/total:.1f}%)')
print(f'  Invalid           : {n_i:>8,}  ({100*n_i/total:.1f}%)')
print()
print('  Confidence breakdown:')
for conf, cnt in out_df['confidence'].value_counts().items():
    print(f'    {conf:<25} {cnt:>6,}  ({100*cnt/total:.1f}%)')
print()
high_c = out_df[out_df['confidence'] == 'high']
if len(high_c):
    print(f'  Avg city score (high-conf) : {high_c["city_score"].mean():.1f}')
print(f'  TF-IDF pass time  : {t_fuzzy:.2f}s')
print(f'  Total elapsed     : {elapsed:.1f}s')
print('═' * 80)
print()
print('── Verified (first 10 rows) ──')
display(verified_df.head(10))
print()
print('── Invalid (all rows) ──')
display(invalid_df)


## Cell 12 — One-shot re-run helper

Re-runs the entire pipeline after changing config in Cell 1.
Does not re-run Cells 3–5 unless reference files change.


In [ ]:
def run_pipeline(
    input_file     = INPUT_FILE,
    dim_path       = DIM_LOCATION,
    alias_path     = ALIAS_FILE,
    out_verified   = OUT_VERIFIED,
    out_invalid    = OUT_INVALID,
    max_rows       = MAX_ROWS,
    use_api        = USE_API,
    api_query_mode = API_QUERY_MODE,
):
    t0 = time.time()
    _dim, _alias, _cities, _provinces, _regions, _barangays = load_reference(dim_path, alias_path)
    _re, _rmap = build_alias_regex(_alias)

    df = pd.read_excel(input_file)
    if max_rows:
        df = df.iloc[:max_rows]
    col = df.columns[0]
    df['address_normalized'] = df[col].apply(lambda x: normalize_alias(str(x), _re, _rmap))

    recs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='TF-IDF match', unit='row'):
        rec = {
            'original_address': row[col],
            'address_normalized': row['address_normalized'],
            'api_status': 'pending' if use_api else 'skipped',
            'osm_city': None, 'osm_province': None, 'osm_region': None,
            'osm_display': None, 'osm_lat': None, 'osm_lon': None,
        }
        rec.update(match_address(row['address_normalized'], _dim, _cities, _provinces, _regions, _barangays))
        recs.append(rec)

    if use_api:
        hi, lo = [], recs
        if lo:
            lo = api_verify_batch(lo, _dim, _cities, _provinces, _regions, query_mode=api_query_mode)
    else:
        hi = [r for r in recs if not r['needs_api']]
        lo = [r for r in recs if r['needs_api']]

    merged = pd.DataFrame(hi + lo)

    if use_api:
        merged['is_valid'] = (
            merged['api_status'].isin(['matched', 'province_only']) &
            (merged['city_name'].notna() | merged['province_name'].notna())
        )
    else:
        merged['is_valid'] = (
            merged['city_name'].notna() |
            (merged['province_name'].notna() & (merged['province_score'] >= PROV_SCORE_HIGH))
        )

    VERIFIED_COLS = [
        'original_address', 'address_normalized',
        'barangay_name', 'barangay_code',
        'city_name', 'city_code',
        'province_name', 'province_code',
        'region_name', 'region_code',
        'psgc_10_digit', 'confidence', 'city_score', 'province_score',
        'osm_display', 'osm_lat', 'osm_lon',
    ]
    INVALID_COLS = [
        'original_address', 'address_normalized',
        'confidence', 'city_score', 'province_score',
        'osm_display', 'api_status',
    ]
    v_df = merged[merged['is_valid']][[c for c in VERIFIED_COLS if c in merged.columns]].reset_index(drop=True)
    i_df = merged[~merged['is_valid']][[c for c in INVALID_COLS  if c in merged.columns]].reset_index(drop=True)

    write_xlsx(v_df, out_verified, 'Verified', '#00B050')
    write_xlsx(i_df, out_invalid,  'Invalid',  '#FF0000', font_color='#FFFFFF')

    elapsed = time.time() - t0
    log.info(f'Done in {elapsed:.1f}s  |  Verified: {len(v_df):,}  Invalid: {len(i_df):,}')
    return v_df, i_df


# Uncomment to run:
# verified_df, invalid_df = run_pipeline()
# verified_df, invalid_df = run_pipeline(api_query_mode='strict')
# verified_df, invalid_df = run_pipeline(use_api=False)
